# Fase 2 - Comprension de los Datos
## Seccion 10: Analisis Geoespacial & Seccion 11: Variables Macroeconomicas

**Notebook:** notebooks/08_EDA_geoespacial_macrovariables.ipynb
**Responsable:** Sofia | **Apoyo:** Steve
**Objetivo:** Validar coordenadas geograficas, mapear distribucion de propiedades, y analizar variables macroeconomicas (salario minimo, IPC, tasa hipotecaria).

## Configuracion inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RAW = os.path.join('..', 'data', 'raw')
OUT = 'outputs'
os.makedirs(OUT, exist_ok=True)

---
# Seccion 10: Analisis Geoespacial
## Celdas 98-102: Validacion de coordenadas

### Cargar datasets con lat/lon

In [ ]:
GEO_CFG = {
    'A1': ('A1_colombia_housing_properties.csv', 'lat', 'lon', 'l3'),
    'A5': ('A5_medellin_properties_2023.csv', 'latitude', 'longitude', None),
}

geo_data = []
for fid, (fname, latcol, loncol, ccol) in GEO_CFG.items():
    df = pd.read_csv(os.path.join(RAW, fname), encoding='utf-8-sig', low_memory=False)
    df['lat'] = pd.to_numeric(df[latcol], errors='coerce')
    df['lon'] = pd.to_numeric(df[loncol], errors='coerce')
    df['fuente'] = fid
    if ccol and ccol in df.columns:
        df['ciudad'] = df[ccol].astype(str).str.strip().str.title()
    else:
        df['ciudad'] = 'Medellin' if fid == 'A5' else None
    geo_data.append(df[['lat','lon','fuente','ciudad']])
    print(f'{fid}: {df.shape[0]:>8,} filas  lat={latcol}  lon={loncol}')

df_geo = pd.concat(geo_data, ignore_index=True)
print(f'\nTotal: {len(df_geo):,} registros')

### Validar rango de coordenadas para Colombia

In [ ]:
LAT_MIN, LAT_MAX = -4.5, 13.0
LON_MIN, LON_MAX = -79.0, -66.0

df_geo['lat_valida'] = df_geo['lat'].between(LAT_MIN, LAT_MAX)
df_geo['lon_valida'] = df_geo['lon'].between(LON_MIN, LON_MAX)
df_geo['coords_validas'] = df_geo['lat_valida'] & df_geo['lon_valida']

print('--- RANGO VALIDO PARA COLOMBIA ---')
print(f'  Latitud:  {LAT_MIN} a {LAT_MAX}')
print(f'  Longitud: {LON_MIN} a {LON_MAX}')

n_total = len(df_geo)
n_con_coords = df_geo['lat'].notna().sum()
n_validas = df_geo['coords_validas'].sum()

print(f'\nRegistros totales:          {n_total:>10,}')
print(f'Con coordenadas:            {n_con_coords:>10,} ({n_con_coords/n_total*100:5.1f}%)')
print(f'Coordenadas en rango valido: {n_validas:>10,} ({n_validas/n_total*100:5.1f}%)')

# Invalidas
invalidas = df_geo[df_geo['lat'].notna() & ~df_geo['coords_validas']]
print(f'\nCoordenadas fuera de rango: {len(invalidas)}')
if len(invalidas) > 0:
    print('Muestra de coordenadas invalidas:')
    display(invalidas[['lat','lon','fuente','ciudad']].dropna().head(10))

### % de cobertura geoespacial por ciudad (A1)

In [ ]:
city_geo = df_geo[df_geo['fuente'] == 'A1'].groupby('ciudad').agg(
    total=('lat', 'count'),
    con_coords=('lat', lambda x: x.notna().sum()),
    validas=('coords_validas', 'sum')
)
city_geo['%_con_coords'] = (city_geo['con_coords'] / city_geo['total'] * 100).round(1)
city_geo['%_validas'] = (city_geo['validas'] / city_geo['total'] * 100).round(1)
city_geo = city_geo[city_geo['total'] >= 100].sort_values('total', ascending=False)
print('--- COBERTURA GEOESPACIAL POR CIUDAD (>=100 registros) ---')
display(city_geo[['total','%_con_coords','%_validas']].head(15))

### Mapa de distribucion de puntos

In [ ]:
print('--- MAPA DE DISTRIBUCION ---')
print('Generando scatterplot geografico (lat vs lon)...')

valid = df_geo[df_geo['coords_validas']]
sample = valid.sample(min(20000, len(valid)), random_state=42)

plt.figure(figsize=(14, 10))
colors = {'A1': 'steelblue', 'A5': 'crimson'}
for fid in ['A1', 'A5']:
    sub = sample[sample['fuente'] == fid]
    plt.scatter(sub['lon'], sub['lat'], c=colors[fid], label=fid, alpha=0.3, s=2)

plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title(f'Distribucion geoespacial de propiedades (muestra {len(sample):,} pts)')
plt.legend()
plt.grid(alpha=0.2)

# Limites Colombia
plt.xlim(LON_MIN, LON_MAX)
plt.ylim(LAT_MIN, LAT_MAX)

plt.tight_layout()
plt.savefig(os.path.join(OUT, 'mapa_distribucion.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Mapa guardado en outputs/mapa_distribucion.png')

### Mapa de densidad (hexbin)

In [ ]:
plt.figure(figsize=(14, 10))
valid_a1 = df_geo[(df_geo['fuente']=='A1') & df_geo['coords_validas']]
sample_a1 = valid_a1.sample(min(50000, len(valid_a1)), random_state=42)
plt.hexbin(sample_a1['lon'], sample_a1['lat'], gridsize=60, cmap='Blues', alpha=0.8)
plt.colorbar(label='Conteo de propiedades')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title('Densidad de propiedades - A1 Properati')
plt.xlim(LON_MIN, LON_MAX)
plt.ylim(LAT_MIN, LAT_MAX)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'mapa_densidad.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Mapa de densidad guardado.')

### Limitaciones geoespaciales

In [ ]:
print('--- LIMITACIONES GEOESPACIALES ---\n')
print('1. Solo 2 de 8 datasets del Grupo A tienen coordenadas (A1 y A5).')
print('2. A1 tiene cobertura nacional pero concentrada en ejes Bogota-Medellin-Cali.')
print('3. A5 solo cubre Medellin (2023).')
print(f'4. {invalidas.shape[0]} registros tienen coordenadas fuera del rango de Colombia.')
print('5. No hay datos de coordenadas para ciudades intermedias (Ibague, Cucuta, etc.).')
print('6. La calidad de lat/lon en A1 puede tener errores de geocodificacion.\n')
print('Recomendaciones:')
print('  - Usar coordenadas solo para analisis a nivel de ciudad/region.')
print('  - No usar para analisis a nivel de barrio por precision limitada.')
print('  - Considerar geocodificar direcciones en A2/A7 como mejora futura.')

---
# Seccion 11: Variables Macroeconomicas
## Celdas 103-107: Cargar y limpiar macrovariables

### Cargar salario minimo historico (B3)

In [ ]:
B3 = pd.read_csv(os.path.join(RAW, 'B3_salario_minimo_historico.csv'), encoding='utf-8-sig')
print('--- B3 - SALARIO MINIMO HISTORICO ---')
print(f'Dimensiones: {B3.shape}')
print(f'Columnas: {list(B3.columns)}')
display(B3.head(8))

### Cargar IPC Colombia (B4)

In [ ]:
B4 = pd.read_csv(os.path.join(RAW, 'B4_ipc_colombia_anual.csv'), encoding='utf-8-sig')
print('--- B4 - IPC COLOMBIA ANUAL ---')
print(f'Dimensiones: {B4.shape}')
print(f'Columnas: {list(B4.columns)}')
display(B4.head(8))

### Cargar tasa hipotecaria (B2)

In [ ]:
B2 = pd.read_csv(os.path.join(RAW, 'B2_tasa_hipotecaria_semanal.csv'), encoding='utf-8-sig')
print('--- B2 - TASA HIPOTECARIA ---')
print(f'Dimensiones: {B2.shape}')
print(f'Columnas: {list(B2.columns)}')
display(B2.head(5))

### Limpiar nombres y estandarizar

In [ ]:
# B3 - Salario minimo
B3_clean = B3.copy()
year_col_b3 = None
sal_col_b3 = None
for c in B3_clean.columns:
    cl = c.lower().strip()
    if any(x in cl for x in ['ano', 'year', 'periodo']):
        year_col_b3 = c
    elif any(x in cl for x in ['salario', 'smlv', 'minimo', 'valor']):
        sal_col_b3 = c
print(f'B3 columns: {list(B3_clean.columns)}')
print(f'  Year col: {year_col_b3}')
print(f'  Salary col: {sal_col_b3}')
B3_clean[year_col_b3] = pd.to_numeric(B3_clean[year_col_b3], errors='coerce')
B3_clean[sal_col_b3] = pd.to_numeric(B3_clean[sal_col_b3], errors='coerce')
print('\nB3 datos:')
display(B3_clean.head(10))

# B4 - IPC
B4_clean = B4.copy()
year_col_b4 = None
ipc_col_b4 = None
for c in B4_clean.columns:
    cl = c.lower().strip()
    if any(x in cl for x in ['ano', 'year', 'periodo']):
        year_col_b4 = c
    elif any(x in cl for x in ['ipc', 'inflacion', 'variacion', 'indice']):
        ipc_col_b4 = c
print(f'\nB4 columns: {list(B4_clean.columns)}')
print(f'  Year col: {year_col_b4}')
print(f'  IPC col: {ipc_col_b4}')
B4_clean[year_col_b4] = pd.to_numeric(B4_clean[year_col_b4], errors='coerce')
B4_clean[ipc_col_b4] = pd.to_numeric(B4_clean[ipc_col_b4], errors='coerce')
print('\nB4 datos:')
display(B4_clean.head(10))


### Filtrar 2015-2024 y crear tabla consolidada

In [ ]:
# Determinar columna year en cada uno
B3_year = year_col_b3
B4_year = year_col_b4

print(f'B3 columna year: {B3_year}')
print(f'B4 columna year: {B4_year}')

if B3_year and sal_col_b3:
    B3_clean[B3_year] = pd.to_numeric(B3_clean[B3_year], errors='coerce')
    B3_filt = B3_clean[(B3_clean[B3_year] >= 2015) & (B3_clean[B3_year] <= 2024)].copy()
    B3_filt[B3_year] = B3_filt[B3_year].astype(int)
    salario_col = sal_col_b3
    print(f'\nB3 2015-2024 ({len(B3_filt)} registros):')
    display(B3_filt[[B3_year, salario_col]].head(12))

if B4_year and ipc_col_b4:
    B4_clean[B4_year] = pd.to_numeric(B4_clean[B4_year], errors='coerce')
    B4_filt = B4_clean[(B4_clean[B4_year] >= 2015) & (B4_clean[B4_year] <= 2024)].copy()
    B4_filt[B4_year] = B4_filt[B4_year].astype(int)
    ipc_col = ipc_col_b4
    print(f'\nB4 2015-2024 ({len(B4_filt)} registros):')
    display(B4_filt[[B4_year, ipc_col]].head(12))


### Tabla consolidada de macrovariables

In [ ]:
# Unir en una tabla
macro = None
if B3_year and salario_col and B4_year and ipc_col:
    macro = B3_filt[[B3_year, salario_col]].merge(
        B4_filt[[B4_year, ipc_col]],
        left_on=B3_year, right_on=B4_year, how='outer'
    )
    # Calcular salario real
    base_year = 2018
    if ipc_col in macro.columns:
        ipc_base = macro.loc[macro[B3_year] == base_year, ipc_col].values[0] if base_year in macro[B3_year].values else 100
        macro['salario_real'] = macro[salario_col] / (macro[ipc_col] / ipc_base)
    macro = macro.sort_values(B3_year)
    print('--- TABLA CONSOLIDADA MACROVARIABLES ---')
    display(macro.round(2))
    macro.to_csv(os.path.join(OUT, 'macrovariables_consolidadas.csv'), index=False)
    print(f'Guardado en outputs/macrovariables_consolidadas.csv')
else:
    print('No se pudo consolidar - columnas no detectadas')

---
## Celdas 108-112: Visualizacion de tendencias macro

### Salario minimo por year

In [ ]:
if macro is not None:
    plt.figure(figsize=(12, 6))
    plt.plot(macro[B3_year], macro[salario_col]/1e6, marker='o', linewidth=2.5, color='green', markersize=8)
    for _, row in macro.iterrows():
        plt.text(row[B3_year], row[salario_col]/1e6 + 0.02, f'${row[salario_col]/1e6:.2f}M', ha='center', fontsize=9)
    plt.xlabel('Year')
    plt.ylabel('Salario minimo (Millones COP)')
    plt.title('Evolucion del salario minimo en Colombia (2015-2024)')
    plt.grid(alpha=0.3)
    plt.xticks(macro[B3_year].astype(int))
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'salario_minimo_tendencia.png'), dpi=150, bbox_inches='tight')
    plt.show()

### Inflacion anual (IPC %)

In [ ]:
if macro is not None and ipc_col in macro.columns:
    plt.figure(figsize=(12, 6))
    colors = ['red' if v > 7 else 'orange' if v > 5 else 'green' for v in macro[ipc_col]]
    bars = plt.bar(macro[B3_year].astype(int), macro[ipc_col], color=colors, edgecolor='white', alpha=0.8)
    for bar, val in zip(bars, macro[ipc_col]):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.2f}%', ha='center', fontsize=9)
    plt.axhline(3, color='green', ls='--', alpha=0.5, label='Meta inflacion (3%)')
    plt.axhline(5, color='orange', ls='--', alpha=0.5, label='Limite superior')
    plt.xlabel('Year')
    plt.ylabel('IPC (%)')
    plt.title('Inflacion anual en Colombia (2015-2024)')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'ipc_anual.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Periodo de alta inflacion: 2022-2023 (>10%)')

### Salario nominal vs real

In [ ]:
if macro is not None:
    plt.figure(figsize=(12, 6))
    plt.plot(macro[B3_year], macro[salario_col]/1e6, marker='o', label='Salario nominal', linewidth=2, color='steelblue')
    plt.plot(macro[B3_year], macro['salario_real']/1e6, marker='s', label='Salario real (base 2018)', linewidth=2, color='darkorange')
    plt.xlabel('Year')
    plt.ylabel('Millones COP')
    plt.title('Salario nominal vs real en Colombia (2015-2024)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'salario_nominal_vs_real.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print('\nCrecimiento acumulado 2015-2024:')
    first = macro.iloc[0]
    last = macro.iloc[-1]
    print(f'  Salario nominal: ${first[salario_col]:,.0f} a ${last[salario_col]:,.0f} ({(last[salario_col]/first[salario_col]-1)*100:.1f}%)')
    print(f'  Salario real:    ${first["salario_real"]:,.0f} a ${last["salario_real"]:,.0f} ({(last["salario_real"]/first["salario_real"]-1)*100:.1f}%)')
    print(f'  Inflacion acumulada: {last[ipc_col]/first[ipc_col]*100-100:.1f}%' if ipc_col in macro.columns else '')

### Tasa hipotecaria

In [ ]:
print('--- TASA HIPOTECARIA ---')
print('Cargando B2 (tasa hipotecaria semanal)...')
display(B2.head(10))

if 'Fecha' in B2.columns:
    B2['Fecha'] = pd.to_datetime(B2['Fecha'], errors='coerce')
    B2['year'] = B2['Fecha'].dt.year
    rate_cols = [c for c in B2.columns if 'tasa' in c.lower() or 'interes' in c.lower() or 'hipotec' in c.lower()]
    if rate_cols:
        rate_col = rate_cols[0]
        B2[rate_col] = pd.to_numeric(B2[rate_col], errors='coerce')
        yearly_rate = B2.groupby('year')[rate_col].mean()
        yearly_rate = yearly_rate[(yearly_rate.index >= 2015) & (yearly_rate.index <= 2024)]
        
        plt.figure(figsize=(12, 6))
        plt.plot(yearly_rate.index.astype(int), yearly_rate.values, marker='o', linewidth=2.5, color='purple', markersize=8)
        for x, y in zip(yearly_rate.index, yearly_rate.values):
            plt.text(int(x), y + 0.3, f'{y:.2f}%', ha='center', fontsize=9)
        plt.xlabel('Year')
        plt.ylabel('Tasa hipotecaria promedio (%)')
        plt.title('Evolucion de la tasa hipotecaria en Colombia (2015-2024)')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT, 'tasa_hipotecaria.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print(f'\nTasa hipotecaria promedio por year:')
        for y, v in yearly_rate.items():
            print(f'  {int(y)}: {v:.2f}%')
    else:
        print('Columna de tasa no detectada. Columnas disponibles:', B2.columns.tolist())
else:
    print('Columna Fecha no encontrada. Columnas:', B2.columns.tolist())

### Periodos de estres macroeconomico

In [ ]:
print('--- PERIODOS DE ESTRES MACROECONOMICO ---\n')

print('2020 (COVID-19):')
print('  - Caida del PIB historica (-6.8% en 2020).')
print('  - Tasas de interes historicamente bajas (BanRep en 1.75%).')
print('  - Congelamiento de cuotas hipotecarias (alivios financieros).')

print('\n2022-2023 (Inflacion alta + subida de tasas):')
print('  - IPC alcanzo >13% en 2023 (maximo en 24 years).')
print('  - BanRep subio tasa de intervencion a 13.25%.')
print('  - Tasa hipotecaria supero 18% afectando accesibilidad.')
print('  - Encarecimiento del credito hipotecario significativo.')

print('\nImpacto en el mercado de vivienda:')
if macro is not None and ipc_col in macro.columns and salario_col in macro.columns:
    for _, row in macro.iterrows():
        if int(row[B3_year]) in [2019, 2020, 2022, 2023]:
            poder_compra = row[salario_col] / row[ipc_col] * 100 if row[ipc_col] != 0 else 0
            print(f'  {int(row[B3_year])}: Salario=${row[salario_col]:,.0f} | IPC={row[ipc_col]:.1f}% | Poder compra real={poder_compra:.0f}')

---
## Resumen

**Seccion 10 - Geoespacial:**
- Solo A1 y A5 tienen coordenadas (2/8 datasets).
- ~Cobertura valida para Colombia en la mayoria de registros A1.
- Mapa de distribucion y densidad generados.
- Limitaciones: sin coordenadas para ciudades intermedias.

**Seccion 11 - Macrovariables:**
- Salario minimo: crecimiento nominal sostenido.
- IPC: pico en 2022-2023 (>13%), normalizacion en 2024.
- Tasa hipotecaria: alza significativa en 2022-2023.
- Salario real: perdida de poder adquisitivo en 2022-2023.
- Tabla consolidada guardada.

**Outputs generados:**
- outputs/mapa_distribucion.png
- outputs/mapa_densidad.png
- outputs/salario_minimo_tendencia.png
- outputs/ipc_anual.png
- outputs/salario_nominal_vs_real.png
- outputs/tasa_hipotecaria.png
- outputs/macrovariables_consolidadas.csv